In [1]:
import re
import pandas as pd
import os

In [2]:
def split_topics(topic_string):
    # Split the string by comma and strip extra whitespace from each topic
    print([topic.strip() for topic in topic_string.split(',')]) 

In [3]:
def split_numbered_topics(topic_string):
    # This regex looks for a number followed by a period and any whitespace, then captures the rest of the line.
    topics = re.findall(r'\d+\.\s*(.*)', topic_string)
    print([topic.strip() for topic in topics])

In [4]:
def extract_topics(topic_string):
    # Try to extract topics that are wrapped in bold (** ... **)
    topics = re.findall(r'\d+\.\s*\*\*(.*?)\*\*', topic_string)
    if topics:
        topics = [topic.strip() for topic in topics]
    else:
        # Fallback: extract text after the number and period up to a hyphen (if present) or end of line
        topics = re.findall(r'\d+\.\s*(.*?)(?:\s*-\s*.*|$)', topic_string)
        topics = [topic.strip() for topic in topics]
    
    # Print the extracted topics list
    print(topics)

In [5]:
# load merged df with citations and topics
folder_path = '../../tcm/collected_dataset/outputs/3after_missing_topics_broad_topics'
df = pd.read_excel(os.path.join(folder_path, 'topics_citation_full_dataset_glmb.xlsx'))
df

,Index,File_Name,Paper_Name,Keywords,Topics,Journal_Name,Published_Year,Country,Continent,URL,...,1986,1985,1984,1983,1982,1981,1980,ground_truth_topics,lda_topics,broad_topics
0,0,(ASCE)0733-9364(1986)112_3(346).pdf,RESOURCE MANAGEMENT IN CONSTRUCTION,[],"['Resource management', 'Construction industry...",Journal of construction engineering and manage...,1986,United States,North America,https://scholar.google.com/scholar?hl=en&as_sd...,...,0,0,0,0,0,0,0,"['Resource planning', 'cost control', 'constru...","['analysis', 'earliest', 'resources', 'activit...","['Resource Management in Construction', 'Const..."
1,2,(ASCE)0742-597X(2005)21_1(2).pdf,Competency-Based Model for Predicting Construc...,['Human factors; Professional development; Pro...,"['Construction project management', 'Competenc...",Journal of Management in Engineering,2005,United Kingdom,Europe,https://scholar.google.com/scholar?hl=en&as_sd...,...,0,0,0,0,0,0,0,"['Competency-based performance prediction', 'c...","['role', 'competencies', 'analysis', 'key', 'd...","['Project Management Practices', 'Human Resour..."
2,3,(ASCE)0887-3801(2006)20_3(165).pdf,Multi-Agent Framework for General-Purpose Situ...,['Models; Simulation; Construction management;...,"['Multi-agent framework', 'Situational simulat...",Journal of Computing in Civil Engineering,2006,United States,North America,https://scholar.google.com/scholar?hl=en&as_sd...,...,0,0,0,0,0,0,0,"['Multi-agent systems', 'situational simulatio...","['cm', 'events', 'operators', 'activities', 't...","['Education and Training', 'Project Management..."
3,4,(ASCE)1532-6748(2001)1_2(17).pdf,Construction Management Practices Are Slowly C...,[],"['Strategic planning', 'Construction managemen...",Leadership and Management in Engineering,2001,NaN,4,https://scholar.google.com/scholar?hl=en&as_sd...,...,0,0,0,0,0,0,0,['Strategic management in construction industr...,"['500', 'strategic management', 'based', 'mana...","['Strategic Management in Construction', 'Cons..."
4,5,(ASCE)CO.1943-7862.0000100.pdf,Managerial Competencies of Female and Male Con...,['Women; Discrimination; Workplace diversity; ...,"['Female project managers', 'Managerial compet...",Journal of Construction Engineering and Manage...,2009,United States,North America,https://scholar.google.com/scholar?hl=en&as_sd...,...,0,0,0,0,0,0,0,"['Gender representation in construction', 'Man...","['competencies', 'focus', 'industry', '2009', ...","['Human Resource Management', 'Construction Ec..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
646,743,Texto.11.ConstructionQuality.pdf,Construction Quality Management: Principles an...,"['Manufacturing', 'Marketing', 'R&D and Engine...","['Quality management', 'Construction organisat...",NaN,2012,NaN,743,https://scholar.google.com/scholar?hl=en&as_sd...,...,0,0,0,0,0,0,0,"['Quality Management', 'Total Quality Manageme...","['learning', 'implementation', 'business', 'ac...","['Quality Control and Assurance', 'Organizatio..."
647,744,Understanding the early stages of the innovati...,Understanding the early stages of the innovati...,"['Communication', 'critical perspective', 'dif...","['awareness', 'influence', 'communication netw...",Construction Management and Economics,2011,United Kingdom,Europe,https://scholar.google.com/scholar?hl=en&as_sd...,...,0,0,0,0,0,0,0,"['Innovation diffusion', 'awareness', 'influen...","['network', 'perspective', 'research', 'constr...",['Communication and Collaboration in Construct...
648,745,Use of attitude congruence to identify safety ...,Use of attitude congruence to identify safety ...,"['Safety', 'small business.']","['Attitude congruence', 'Safety interventions'...",Construction Management and Economics,2011,United States,North America,https://scholar.google.com/scholar?hl=en&as_sd...,...,0,0,0,0,0,0,0,"['Construction industry safety', 'Occupational...","['attitudes', 'safety attitude', 'intervention...","['Organizational Behavior and Culture', 'Const...

# 1. Lexical (String-Based) Similarity using Jaccard Similarity

Step 1: Convert String Representation to Python List
We can use ast.literal_eval to safely parse the string into a list:

Step 2: Tokenize Each Phrase and Build a Set of Words
We want to compare sets of words across all phrases in each row. For example,

Ground truth might have "Resource planning" which we want to split into tokens ["resource", "planning"].
The LDA topics might have just the token "resources", etc.
We can use a simple regex-based tokenizer that lowercases words and removes punctuation:

Then, for an entire list of phrases (like ["Resource planning", "cost control"]), we aggregate all tokens into a single set:

Step 3: Define a Jaccard Similarity Function
Jaccard Similarity between two sets A and B is defined as:

J(A, B) = \frac{|A \cap B|}{|A \cup B|}

Pros:

Simple to implement and interpret.
Does not require complex models or large corpora.

Cons:

Fails to capture semantic similarity when words differ but mean similar things (e.g. “cost control” vs. “budget limitations”).

In [6]:
import ast
import re

def parse_list_from_string(list_str):
    """
    Parse a string representation of a list into a Python list.
    """
    return ast.literal_eval(list_str)

def tokenize_phrase(phrase):
    """
    Tokenize a string (phrase) into words using a simple regex.
    - Convert to lowercase
    - Extract alphanumeric tokens
    """
    return re.findall(r"\w+", phrase.lower())

def get_token_set(topics_list):
    """
    Convert a list of topic phrases into a set of unique tokens.
    """
    tokens = set()
    for phrase in topics_list:
        tokens.update(tokenize_phrase(phrase))
    return tokens

def jaccard_similarity(str_list1, str_list2):
    """
    Compute the Jaccard similarity between two string-represented lists.
    Each list is turned into a set of tokens before calculating overlap.
    """
    # 1. Parse from string to list
    list1 = parse_list_from_string(str_list1)
    list2 = parse_list_from_string(str_list2)
    
    # 2. Convert each list of phrases to a set of tokens
    # set1 = get_token_set(list1)
    # set2 = get_token_set(list2)
    set1 = set(list1)
    set2 = set(list2)
    
    # 3. Calculate Jaccard similarity
    intersection = set1.intersection(set2)
    union = set1.union(set2)
    
    if len(union) == 0:
        return 0.0  # avoid division by zero if both sets are empty
    return len(intersection) / len(union)


In [7]:
df['jaccard_similarity_lda'] = df.apply(
    lambda row: jaccard_similarity(row['ground_truth_topics'], row['lda_topics']), 
    axis=1
)

In [8]:
df['jaccard_similarity_lda'].mean()

0.010040205983491523

In [9]:
df['jaccard_similarity_llm'] = df.apply(
    lambda row: jaccard_similarity(row['ground_truth_topics'], row['Topics']), 
    axis=1
)

In [10]:
df['jaccard_similarity_llm'].mean()

0.07319375896314322

In [11]:
(df['jaccard_similarity_llm'].mean() - df['jaccard_similarity_lda'].mean()) / df['jaccard_similarity_lda'].mean()

6.290065471115941

## BLEU and ROUGE

In [18]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

# Initialize the ROUGE scorer
scorer = rouge_scorer.RougeScorer(['rouge3', 'rouge2', 'rougeL'], use_stemmer=True)

def calculate_scores(row):
    """Calculate BLEU and ROUGE scores comparing LDA and LLM topics
       to the ground-truth topics in a given DataFrame row."""
    ground_truth =  parse_list_from_string(row["ground_truth_topics"])
    lda_topics = parse_list_from_string(row["lda_topics"])
    llm_topics = parse_list_from_string(row["Topics"])

    
    # ========== BLEU SCORES ==========
    # Treat each topic as a single 'token'. We provide ground_truth in a list 
    # since BLEU expects a list of reference sequences. 
    smooth_fn = SmoothingFunction().method1
    bleu_lda = sentence_bleu([ground_truth], lda_topics, smoothing_function=smooth_fn)
    bleu_llm = sentence_bleu([ground_truth], llm_topics, smoothing_function=smooth_fn)
    
    # ========== ROUGE SCORES ==========
    # Join topic lists into space-separated strings
    gt_str = " ".join(ground_truth)
    lda_str = " ".join(lda_topics)
    llm_str = " ".join(llm_topics)
    
    rouge_lda_scores = scorer.score(gt_str, lda_str)
    rouge_llm_scores = scorer.score(gt_str, llm_str)
    
    return pd.Series({
        'BLEU_LDA': bleu_lda,
        'BLEU_LLM': bleu_llm,
        
        'ROUGE3_LDA': rouge_lda_scores['rouge3'].fmeasure,
        'ROUGE2_LDA': rouge_lda_scores['rouge2'].fmeasure,
        'ROUGE_L_LDA': rouge_lda_scores['rougeL'].fmeasure,
        
        'ROUGE3_LLM': rouge_llm_scores['rouge3'].fmeasure,
        'ROUGE2_LLM': rouge_llm_scores['rouge2'].fmeasure,
        'ROUGE_L_LLM': rouge_llm_scores['rougeL'].fmeasure
    })

# Apply the scoring function to each row
df_scores = df.apply(calculate_scores, axis=1)

# Combine the original DF with the scores side-by-side if you want
# df_with_scores = pd.concat([df, df_scores], axis=1)

In [19]:
df_scores

,BLEU_LDA,BLEU_LLM,ROUGE3_LDA,ROUGE2_LDA,ROUGE_L_LDA,ROUGE3_LLM,ROUGE2_LLM,ROUGE_L_LLM
0,0.000000,0.009233,0.0,0.000000,0.235294,0.092308,0.208955,0.289855
1,0.000000,0.000000,0.0,0.040816,0.313725,0.117647,0.377358,0.472727
2,0.009630,0.000000,0.0,0.037037,0.142857,0.117647,0.264151,0.436364
3,0.000000,0.000000,0.0,0.027027,0.184211,0.000000,0.135135,0.236842
4,0.000000,0.020549,0.0,0.037736,0.218182,0.000000,0.226415,0.290909
...,...,...,...,...,...,...,...,...
646,0.000000,0.000000,0.0,0.000000,0.113208,0.200000,0.384615,0.370370
647,0.011452,0.073173,0.0,0.083333,0.200000,0.142857,0.363636,0.391304
648,0.000000,0.019620,0.0,0.101695,0.229508,0.150943,0.400000,0.385965
649,0.000000,0.000000,0.0,0.000000,0.166667,0.235294,0.333333,0.421053


In [20]:
df_scores.mean()

BLEU_LDA       0.002370
BLEU_LLM       0.019285
ROUGE3_LDA     0.005025
ROUGE2_LDA     0.065817
ROUGE_L_LDA    0.212660
ROUGE3_LLM     0.161500
ROUGE2_LLM     0.307252
ROUGE_L_LLM    0.424436
dtype: float64

In [24]:
# df_with_scores
df[['BLEU_LDA', 'BLEU_LLM', 'ROUGE3_LDA', 'ROUGE2_LDA', 'ROUGE3_LLM', 'ROUGE2_LLM']] = df_scores[['BLEU_LDA', 'BLEU_LLM', 'ROUGE3_LDA', 'ROUGE2_LDA', 'ROUGE3_LLM', 'ROUGE2_LLM']]

In [26]:
df

,Index,File_Name,Paper_Name,Keywords,Topics,Journal_Name,Published_Year,Country,Continent,URL,...,lda_topics,broad_topics,jaccard_similarity_lda,jaccard_similarity_llm,BLEU_LDA,BLEU_LLM,ROUGE3_LDA,ROUGE2_LDA,ROUGE3_LLM,ROUGE2_LLM
0,0,(ASCE)0733-9364(1986)112_3(346).pdf,RESOURCE MANAGEMENT IN CONSTRUCTION,[],"['Resource management', 'Construction industry...",Journal of construction engineering and manage...,1986,United States,North America,https://scholar.google.com/scholar?hl=en&as_sd...,...,"['analysis', 'earliest', 'resources', 'activit...","['Resource Management in Construction', 'Const...",0.000000,0.071429,0.000000,0.009233,0.0,0.000000,0.092308,0.208955
1,2,(ASCE)0742-597X(2005)21_1(2).pdf,Competency-Based Model for Predicting Construc...,['Human factors; Professional development; Pro...,"['Construction project management', 'Competenc...",Journal of Management in Engineering,2005,United Kingdom,Europe,https://scholar.google.com/scholar?hl=en&as_sd...,...,"['role', 'competencies', 'analysis', 'key', 'd...","['Project Management Practices', 'Human Resour...",0.000000,0.000000,0.000000,0.000000,0.0,0.040816,0.117647,0.377358
2,3,(ASCE)0887-3801(2006)20_3(165).pdf,Multi-Agent Framework for General-Purpose Situ...,['Models; Simulation; Construction management;...,"['Multi-agent framework', 'Situational simulat...",Journal of Computing in Civil Engineering,2006,United States,North America,https://scholar.google.com/scholar?hl=en&as_sd...,...,"['cm', 'events', 'operators', 'activities', 't...","['Education and Training', 'Project Management...",0.029412,0.000000,0.009630,0.000000,0.0,0.037037,0.117647,0.264151
3,4,(ASCE)1532-6748(2001)1_2(17).pdf,Construction Management Practices Are Slowly C...,[],"['Strategic planning', 'Construction managemen...",Leadership and Management in Engineering,2001,NaN,4,https://scholar.google.com/scholar?hl=en&as_sd...,...,"['500', 'strategic management', 'based', 'mana...","['Strategic Management in Construction', 'Cons...",0.000000,0.000000,0.000000,0.000000,0.0,0.027027,0.000000,0.135135
4,5,(ASCE)CO.1943-7862.0000100.pdf,Managerial Competencies of Female and Male Con...,['Women; Discrimination; Workplace diversity; ...,"['Female project managers', 'Managerial compet...",Journal of Construction Engineering and Manage...,2009,United States,North America,https://scholar.google.com/scholar?hl=en&as_sd...,...,"['competencies', 'focus', 'industry', '2009', ...","['Human Resource Management', 'Construction Ec...",0.000000,0.100000,0.000000,0.020549,0.0,0.037736,0.000000,0.226415
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
646,743,Texto.11.ConstructionQuality.pdf,Construction Quality Management: Principles an...,"['Manufacturing', 'Marketing', 'R&D and Engine...","['Quality management', 'Construction organisat...",NaN,2012,NaN,743,https://scholar.google.com/scholar?hl=en&as_sd...,...,"['learning', 'implementation', 'business', 'ac...","['Quality Control and Assurance', 'Organizatio...",0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.200000,0.384615
647,744,Understanding the early stages of the innovati...,Understanding the early stages of the innovati...,"['Communication', 'critical perspective', 'dif...","['awareness', 'influence', 'communication netw...",Construction Management and Economics,2011,United Kingdom,Europe,https://scholar.google.com/scholar?hl=en&as_sd...,...,"['network', 'perspective', 'research', 'constr...",['Communication and Collaboration in Construct...,0.062500,0.210526,0.011452,0.073173,0.0,0.083333,0.142857,0.363636
648,745,Use of attitude congruence to identify safety ...,Use of attitude congruence to identify safety ...,"['Safety', 'small business.']","['Attitude congruence', 'Safety interventions'...",Construction Management and Economics,2011,United States,North America,https://scholar.google.com/scholar?hl=en&as_sd...,...,"['attitudes', 'safety attitude', 'intervention...","['Organizational Behavior and Culture', 'Const...",0.

In [27]:
# save the dataframe back to xlsx file
save_path = '../../tcm/collected_dataset/outputs/4after_eval'
df.to_excel(os.path.join(save_path, 'topics_citation_full_dataset_glmbe.xlsx'), index=False)

# 2. Semantic (Meaning-Based) Similarity

# 2.1 Word-Level Embeddings (Glove + Cosine similarity)

Convert each word into an embedding.
For multi-word phrases like “resource planning,” you might average the embeddings of “resource” and “planning.”
For LDA words, each is already a single token.
Then compare embeddings across the sets using cosine similarity (e.g., for each ground-truth phrase, find its similarity to each LDA topic token).
Compute an average or maximum similarity from each phrase in ground_truth_topics to the tokens in lda_topics to get an overall set-to-set similarity.

Pros:

Captures some semantic relationships.
Straightforward approach if embeddings are available.

Cons:

Single-word embeddings often miss context.
Phrases with multiple words are represented by a simple average or sum, which may be too coarse.

In [9]:
# import gensim.downloader as api
# 
# # This downloads a 100-dimensional GloVe model trained on Wikipedia + Gigaword:
# glove_model = api.load("glove-wiki-gigaword-200")


[====----------------------------------------------] 9.7% 24.6/252.1MB downloaded

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[==========----------------------------------------] 20.4% 51.4/252.1MB downloaded

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[=================---------------------------------] 34.4% 86.7/252.1MB downloaded

...

[========================================----------] 81.4% 205.2/252.1MB downloaded

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[==============================================----] 93.2% 234.9/252.1MB downloaded

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[==================================================] 100.0% 252.1/252.1MB downloaded


# 2.1.1 Tokenization

We’ll define a simple tokenizer that splits each phrase into words. We’ll also lowercase to match GloVe’s vocabulary (which is mostly lowercase):

In [10]:
# defined above

# 2.1.2 Get Average GloVe Embedding for a Phrase

We can convert a phrase (e.g., "Resource planning") into an average embedding of its tokens. We skip tokens that are not in the GloVe vocabulary.

In [10]:
# import numpy as np
# 
# def get_glove_embedding_for_phrase(phrase, glove_model):
#     """
#     Return the average GloVe embedding for all tokens in 'phrase'.
#     If no token is in vocab, return a zero vector.
#     """
#     tokens = tokenize_phrase(phrase)
#     valid_vectors = []
#     for token in tokens:
#         if token in glove_model.key_to_index:  # check if token in GloVe vocab
#             valid_vectors.append(glove_model[token])
#     
#     if not valid_vectors:
#         return np.zeros(glove_model.vector_size)
#     
#     return np.mean(valid_vectors, axis=0)


# 2.1.3 Get Average GloVe Embedding for a List of Phrases

Sometimes you want a single embedding to represent the entire list of topics for easier set-to-set similarity. One approach:

For each phrase in the list, compute an average phrase embedding.
Combine those phrase embeddings by averaging again (or by summing, etc.).

In [28]:
# def set_to_set_glove_similarity(ground_truth_topics, other_topics, glove_model):
#     """
#     For each phrase g in ground_truth_topics:
#       1) Compute embedding of g
#       2) Compute embedding of each topic t in other_topics
#       3) Find maximum cosine similarity: max_{t in T} cos_sim(emb(g), emb(t))
#     
#     Then average these maxima across all g in ground_truth_topics.
#     """
#     if not ground_truth_topics:
#         return 0.0
#     
#     similarities = []
#     
#     # Precompute embeddings for "other_topics" to avoid repeated calculations
#     other_embeddings = [
#         get_glove_embedding_for_phrase(t, glove_model) for t in other_topics
#     ]
#     
#     for g in ground_truth_topics:
#         g_emb = get_glove_embedding_for_phrase(g, glove_model)
#         
#         # If the "other" list is empty or no valid embeddings, similarity is 0
#         if not other_embeddings:
#             similarities.append(0.0)
#             continue
#         
#         # Find the best match
#         best_sim = max(cosine_similarity(g_emb, o_emb) 
#                        for o_emb in other_embeddings)
#         similarities.append(best_sim)
#     
#     # Average the best-match similarities
#     return float(np.mean(similarities))

# def set_to_set_glove_similarity_avg_of_avg(ground_truth_topics, other_topics, glove_model):
#     """
#     Average-of-Averages (AA):
#       1) For each ground-truth phrase g, compute embedding emb(g).
#       2) For each emb(g), compute similarity to each topic embedding in 'other_topics',
#          and take the AVERAGE of those similarities.
#       3) Finally, average these per-g averages across all ground-truth g.
#     """
#     if not ground_truth_topics or not other_topics:
#         return 0.0
# 
#     # Precompute embeddings for the "other_topics"
#     other_embeddings = [get_glove_embedding_for_phrase(t, glove_model) for t in other_topics]
#     # Filter out empty embeddings if no token recognized
#     other_embeddings = [o for o in other_embeddings if np.any(o)]
# 
#     if not other_embeddings:
#         # If we have no valid embeddings in the other set, similarity = 0
#         return 0.0
# 
#     similarities = []
#     for g in ground_truth_topics:
#         g_emb = get_glove_embedding_for_phrase(g, glove_model)
#         if not np.any(g_emb):
#             # If g is empty or not recognized, similarity = 0
#             similarities.append(0.0)
#             continue
# 
#         # Compute similarity of g_emb to each o_emb in the other set, then average
#         sim_list = [cosine_similarity(g_emb, o_emb) for o_emb in other_embeddings]
#         avg_sim = np.mean(sim_list)
#         similarities.append(avg_sim)
# 
#     return float(np.mean(similarities))
# 
# 
# def compute_row_similarities(row, glove_model):
#     gtruth = parse_list_from_string(row["ground_truth_topics"])
#     lda    = parse_list_from_string(row["lda_topics"])
#     llm    = parse_list_from_string(row["Topics"])
#     
#     
#     
#     lda_sim = set_to_set_glove_similarity_avg_of_avg(gtruth, lda, glove_model)
#     llm_sim = set_to_set_glove_similarity_avg_of_avg(gtruth, llm, glove_model)
#     
#     return pd.Series({
#         "lda_set2set_similarity": lda_sim,
#         "llm_set2set_similarity": llm_sim
#     })


# 2.1.4 Compute Cosine Similarity

We want to compare:

lda_topics with ground_truth_topics
llm_topics with ground_truth_topics
We’ll create new columns for each similarity measure.

4.1 Row-by-Row Comparison
For each row in the DataFrame:

Convert the entire ground_truth_topics into one embedding.
Convert lda_topics into one embedding.
Convert llm_topics into one embedding.
Compute cosine similarity with the ground-truth embedding.

In [12]:
# from numpy.linalg import norm
# 
# def cosine_similarity(vec1, vec2):
#     """
#     Compute the cosine similarity between two vectors.
#     """
#     dot_product = np.dot(vec1, vec2)
#     denom = norm(vec1) * norm(vec2)
#     if denom == 0.0:
#         return 0.0
#     return dot_product / denom


In [29]:
# df[["lda_set2set_similarity", "llm_set2set_similarity"]] = df.apply(
#     compute_row_similarities,
#     axis=1,
#     args=(glove_model,)
# )

ValueError: Expected 2D array, got 1D array instead:
array=[ 0.3956105   0.06516501  0.2257      0.15474999  0.20305002  0.113015
 -0.32031     0.10202    -0.31372     0.13711001 -0.0820885   0.3213
  0.78462     0.276035    0.318035    0.015515   -0.1752245  -0.083273
  0.0067855  -0.03878002  0.005495    2.57215    -0.0896      0.379545
 -0.0164595   0.3361105   0.1072575   0.49588     0.411015   -0.02839
  0.01809999 -0.3278      0.0602695   0.144435    0.50464    -0.036959
  0.00967     0.1644475   0.12379001 -0.003245   -0.149533   -0.34109
 -0.0858065  -0.313115    0.04804501  0.24961501 -0.008845   -0.17283499
  0.2573574  -0.0886      0.42101002  0.12669998 -0.13532     0.075544
 -0.31205502 -0.38832     0.21790035 -0.46605     0.245035    0.19510001
  0.601285    0.287855   -0.249605    0.18718499 -0.3386005  -0.03134999
 -0.13581     0.88427997 -0.41149998 -0.34085736 -0.0176931  -0.268525
  0.45357    -0.09967851  0.0996525   0.697675    0.2208125   0.09605449
 -0.0523375  -

In [30]:
# df

,Index,File_Name,Paper_Name,Keywords,Topics,Journal_Name,Published_Year,Country,Continent,URL,...,1982,1981,1980,ground_truth_topics,lda_topics,broad_topics,jaccard_similarity_lda,jaccard_similarity_llm,lda_set2set_similarity,llm_set2set_similarity
0,0,(ASCE)0733-9364(1986)112_3(346).pdf,RESOURCE MANAGEMENT IN CONSTRUCTION,[],"['Resource management', 'Construction industry...",Journal of construction engineering and manage...,1986,United States,North America,https://scholar.google.com/scholar?hl=en&as_sd...,...,0,0,0,"['Resource planning', 'cost control', 'constru...","['analysis', 'earliest', 'resources', 'activit...","['Resource Management in Construction', 'Const...",0.191489,0.325000,0.269095,0.331936
1,2,(ASCE)0742-597X(2005)21_1(2).pdf,Competency-Based Model for Predicting Construc...,['Human factors; Professional development; Pro...,"['Construction project management', 'Competenc...",Journal of Management in Engineering,2005,United Kingdom,Europe,https://scholar.google.com/scholar?hl=en&as_sd...,...,0,0,0,"['Competency-based performance prediction', 'c...","['role', 'competencies', 'analysis', 'key', 'd...","['Project Management Practices', 'Human Resour...",0.303030,0.500000,0.285847,0.333401
2,3,(ASCE)0887-3801(2006)20_3(165).pdf,Multi-Agent Framework for General-Purpose Situ...,['Models; Simulation; Construction management;...,"['Multi-agent framework', 'Situational simulat...",Journal of Computing in Civil Engineering,2006,United States,North America,https://scholar.google.com/scholar?hl=en&as_sd...,...,0,0,0,"['Multi-agent systems', 'situational simulatio...","['cm', 'events', 'operators', 'activities', 't...","['Education and Training', 'Project Management...",0.104167,0.324324,0.279993,0.319568
3,4,(ASCE)1532-6748(2001)1_2(17).pdf,Construction Management Practices Are Slowly C...,[],"['Strategic planning', 'Construction managemen...",Leadership and Management in Engineering,2001,NaN,4,https://scholar.google.com/scholar?hl=en&as_sd...,...,0,0,0,['Strategic management in construction industr...,"['500', 'strategic management', 'based', 'mana...","['Strategic Management in Construction', 'Cons...",0.137931,0.283019,0.289969,0.330932
4,5,(ASCE)CO.1943-7862.0000100.pdf,Managerial Competencies of Female and Male Con...,['Women; Discrimination; Workplace diversity; ...,"['Female project managers', 'Managerial compet...",Journal of Construction Engineering and Manage...,2009,United States,North America,https://scholar.google.com/scholar?hl=en&as_sd...,...,0,0,0,"['Gender representation in construction', 'Man...","['competencies', 'focus', 'industry', '2009', ...","['Human Resource Management', 'Construction Ec...",0.175000,0.411765,0.294849,0.332795
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
646,743,Texto.11.ConstructionQuality.pdf,Construction Quality Management: Principles an...,"['Manufacturing', 'Marketing', 'R&D and Engine...","['Quality management', 'Construction organisat...",NaN,2012,NaN,743,https://scholar.google.com/scholar?hl=en&as_sd...,...,0,0,0,"['Quality Management', 'Total Quality Manageme...","['learning', 'implementation', 'business', 'ac...","['Quality Control and Assurance', 'Organizatio...",0.170732,0.393939,0.282429,0.326590
647,744,Understanding the early stages of the innovati...,Understanding the early stages of the innovati...,"['Communication', 'critical perspective', 'dif...","['awareness', 'influence', 'communication netw...",Construction Management and Economics,2011,United Kingdom,Europe,https://scholar.google.com/scholar?hl=en&as_sd...,...,0,0,0,"['Innovation diffusion', 'awareness', 'influen...","['network', 'perspective', 'research', 'constr...",['Communication and Collaboration in Construct...,0.406250,0.400000,0.295905,0.337595
648,745,Use of attitude congruence to identify safety ...,Use of attitude congruence to identify safety ...,"['Safety', 'small business.']","['Attitude congruence', 'Safety interventions'...",Construction Management a

In [20]:
# df['lda_set2set_similarity'].mean()

0.2802557012168729

In [21]:
# df['llm_set2set_similarity'].mean()

0.3278607433656702

# 2.2 Sentence-Level Embeddings (Transformer-based embeddings)



Pros:

Much better at handling synonyms and capturing context.
Phrasal embeddings are better representations of multi-word concepts than simple bag-of-words or Word2Vec.

Cons:

Requires a pre-trained model.
Slightly more complex computationally.


In [4]:
# import pandas as pd
# import numpy as np
# 
# from sentence_transformers import SentenceTransformer
# from sklearn.metrics.pairwise import cosine_similarity

In [11]:
# model = SentenceTransformer('all-MiniLM-L6-v2')
# def set_to_set_transformer_similarity(ground_truth_list, other_list, model):
#     """
#     Given two lists of strings (e.g., ground_truth_topics and lda_topics),
#     1. Encode each string in ground_truth_list -> embeddings_g
#     2. Encode each string in other_list -> embeddings_t
#     3. For each g in embeddings_g, find the max cosine similarity with embeddings_t
#     4. Average these max similarities
# 
#     Returns a single float similarity score.
#     """
#     # Edge cases
#     if not ground_truth_list or not other_list:
#         return 0.0
#     
#     # Step 1: Embed each phrase in ground_truth_list
#     embeddings_g = model.encode(ground_truth_list, show_progress_bar=False)
#     
#     # Step 2: Embed each phrase in other_list
#     embeddings_t = model.encode(other_list, show_progress_bar=False)
#     
#     # Step 3: Compute pairwise similarity matrix (|G| x |T|)
#     sim_matrix = cosine_similarity(embeddings_g, embeddings_t)
#     
#     # For each ground-truth row g, find its best match in T
#     # sim_matrix[i] is a 1D array of similarities of ground_truth i to all topics in other_list
#     max_sims = sim_matrix.mean(axis=1)  # shape = (|G|,)
#     
#     # Step 4: Average these maximum similarities
#     return float(np.mean(max_sims))


In [6]:
# def compute_row_similarities(row, model):
#     # Extract lists
#     gtruth = parse_list_from_string(row["ground_truth_topics"])
#     lda = parse_list_from_string(row["lda_topics"])
#     llm = parse_list_from_string(row["Topics"])
#     
#     # Compute set-to-set similarity with ground_truth_topics
#     lda_sim = set_to_set_transformer_similarity(gtruth, lda, model)
#     llm_sim = set_to_set_transformer_similarity(gtruth, llm, model)
#     
#     return pd.Series({
#         "lda_contextual_similarity": lda_sim,
#         "llm_contextual_similarity": llm_sim
#     })


In [12]:
# df[["lda_contextual_similarity", "llm_contextual_similarity"]] = df.apply(
#     compute_row_similarities,
#     axis=1,
#     args=(model,)
# )


In [13]:
# df['lda_contextual_similarity'].mean()

0.2174751954968624

In [14]:
# df['llm_contextual_similarity'].mean()

0.28964684651835537

In [15]:
# (df['llm_contextual_similarity'].mean() - df['lda_contextual_similarity'].mean()) / df['lda_contextual_similarity'].mean()

0.3318615295716988

# Analysis

In [28]:
# def combine_topics(df):
#     """
#     Combines all topics from the 'Topics' column of the dataframe into a single set,
#     treating topics as identical regardless of case.
#     
#     Parameters:
#         df (pandas.DataFrame): A dataframe with a 'Topics' column where each entry is a list of topics.
#         column_name (str): The name of the column containing the topics.
#     Returns:
#         set: A set containing all unique topics in lower-case.
#     """
#     all_topics = set()
#     for topics_list in parse_list_from_string(df['lda_topics']):
#         # Convert each topic to lower case and add to the set
#         all_topics.update([topic.lower() for topic in topics_list])
#     return all_topics

In [29]:
# calculate_scores(df)

ValueError: malformed node or string: 0      ['Resource planning', 'cost control', 'constru...
1      ['Competency-based performance prediction', 'c...
2      ['Multi-agent systems', 'situational simulatio...
3      ['Strategic management in construction industr...
4      ['Gender representation in construction', 'Man...
                             ...                        
646    ['Quality Management', 'Total Quality Manageme...
647    ['Innovation diffusion', 'awareness', 'influen...
648    ['Construction industry safety', 'Occupational...
649    ['Build-Operate-Transfer', 'minimum revenue gu...
650    ['Work–family enrichment', 'Australian constru...
Name: ground_truth_topics, Length: 651, dtype: object

In [30]:
# all_topics = set()
# for topics_list in df['lda_topics']:
#     # Convert each topic to lower case and add to the set
#     
#     all_topics.update([topic.lower() for topic in parse_list_from_string(topics_list)])
# len(all_topics)

3538